This notebook is for automatically generating the RMG input files and run.sh files for RMG models. 

In [1]:
from itertools import combinations
import itertools
import os

In [2]:
input_file_part_1 = """
thermolibs = [
'NCSU_2025_revised',
'C1_C2_Fluorine',
'PFCA_thermo',
'primaryThermoLibrary',
'FFCM1(-)',
'halogens',
'CHOF_G4',
'CHOCl_G4',
'CHOBr_G4',
'CHOFCl_G4',
'CHOFBr_G4',
'CHOFClBr_G4',
'DFT_QCI_thermo',
'Fluorine',
'2-BTP_G4',
'thermo_DFT_CCSDTF12_BAC',
'SulfurHaynes',
]


database(
thermoLibraries = thermolibs,
reactionLibraries = ['FFCM1(-)', "ANL_Brown_pdep", "ANL_Brown_pdep_2", 'Brown_difluoromethane_seed', 'halogens_pdep'],    
seedMechanisms = [],
kineticsDepositories = ['training'],
kineticsFamilies = ['default','halogens', "PFAS", 'Disproportionation-Y'],
frequenciesLibraries = ['halogens_G4'],
kineticsEstimator = 'rate rules',
)
"""
   
input_file_part_3 = """    
simulator(
    atol = 1e-16,
    rtol = 1e-08,
    sens_atol = 1e-06,
    sens_rtol = 0.0001,
)


generatedSpeciesConstraints(
    allowed=['input species','seed mechanisms','reaction libraries'],
    maximumCarbonAtoms=8,
    maximumOxygenAtoms=6,
    #maximumHeavyAtoms=24,
    maximumRadicalElectrons=2,
    maximumSingletCarbenes=1,
    maximumCarbeneRadicals=0,
    allowSingletO2 = True,
)

options(
    units = "si",
    generateSeedEachIteration = True,
    generateOutputHTML = False,
    generatePlots = False,
    saveSimulationProfiles = False,
    saveEdgeSpecies = True,
    keepIrreversible = True,
    verboseComments = False,
)
"""    


input_file_part_5 = """

model(
    toleranceMoveToCore = 0.1,
    toleranceBranchReactionToCore=0.001, #using recommended values, which are said to be sufficient
    toleranceInterruptSimulation = 0.1,
    maximumEdgeSpecies = 3e5,
    filterReactions = True,
    filterThreshold = 5e8,
    minCoreSizeForPrune = 50,
    minSpeciesExistIterationsForPrune = 4,
    branchingIndex=0.5, #using recommended values,
    branchingRatioMax=1.0, #using recommended values,
)


pressureDependence(
    method='modified strong collision',
    maximumGrainSize=(0.5,'kcal/mol'),
    minimumNumberOfGrains=250,
    temperatures=(300,2500,'K',8),
    pressures=(0.01,100,'bar',5),
    interpolation=('Chebyshev', 6, 4),
    maximumAtoms=16,
    completedNetworks=['CH2O2', 'CO3'],
)
"""



In [2]:
# this atom_dictionaries should be changed depending on the blend compositions you want to simulate. 

atom_dictionaries = {"R50": {'formula': 'CH4', 'count': {'C': 1, 'H': 4}, 'smiles': 'C'}, 
                     "R290": {'formula': 'C3H8', 'count': {'C': 3, 'H': 8}, 'smiles': 'CCC'}, 
                     "R600a": {'formula': 'C4H10', 'count': {'C': 4, 'H': 10}, 'smiles': 'CC(C)C'}, 

                     
                     "R32": {'formula': 'CH2F2', 'count': {'C': 1, 'H': 2, 'F': 2}, 'smiles': 'C(F)F'},
                     "R134a": {'formula': 'CH2F-CF3', 'count': {'C': 2, 'H': 2, 'F': 4}, 'smiles': 'C(C(F)(F)F)F'},
                     "R152a": {'formula': 'CH3-CHF2', 'count': {'C': 2, 'H': 4, 'F': 2}, 'smiles': 'CC(F)F'},

                     "R1234ze(E)": {'formula': 'CHF=CH-CF3', 'count': {'C': 3, 'H': 2, 'F': 4}},
                     "R1234yf": {'formula': 'CH2=CF-CF3', 'count': {'C': 3, 'H': 2, 'F': 4}},
                     "HFO-1123": {'formula': 'CF2=CHF', 'count': {'C': 2, 'H': 1, 'F': 3}},

                     "CO2": {'formula': 'CO2', 'count': {'C': 1, 'O': 2}},
                     "HF": {'formula': 'HF', 'count': {'H': 1, 'F': 1}}, 
                     "H2O": {'formula': 'H2O', 'count': {'H': 2, 'O': 1}}, 
                    
                    }



def find_stoichiometric_equation(component_1, component_2, ratio_1_to_2): 

    """This function was written with complete stoichiometric combustion in mind, yielding CO2 and H2O
        as products. 

        WARNING: In cases where the number of fluorine atoms exceeds the number of hydrogen atoms in the refrigerant species, 
        carbonyl fluoride (COF2) should be predicted as an additional combustion product, capturing the fluorine atoms not accounted for by HF formation
        (e.g. R1234yf: C3H2F4 + 2.5 (O2 + 3.76 N2) −−→2 CO2 + 2 HF + COF2 + 9.4 N2).   
        Depending on your blend composition, you might have to extend this function so it predicts COF2 as a product. 
    """
    
    (fraction_of_comp_1, fraction_of_comp_2) = ratio_1_to_2
    
    dict_1 = atom_dictionaries[component_1]
    formula_1 = dict_1['formula']
    count_1 = dict_1['count']

    dict_2 = atom_dictionaries[component_2]
    formula_2 = dict_2['formula']
    count_2 = dict_2['count']    

    comp_1_coeff = fraction_of_comp_1 
    comp_2_coeff = fraction_of_comp_2 

    
    CO2_coeff = comp_1_coeff*count_1.get('C', 0) + comp_2_coeff*count_2.get('C', 0)
    HF_coeff = comp_1_coeff*count_1.get('F', 0) + comp_2_coeff*count_2.get('F', 0)
    H2O_coeff = ((comp_1_coeff*count_1.get('H', 0) + comp_2_coeff*count_2.get('H', 0)) - HF_coeff)/2
    O2_coeff = (2*CO2_coeff + H2O_coeff)/2
    N2_coeff = O2_coeff*3.76


    norm_comp_1_coeff = comp_1_coeff/O2_coeff 
    norm_comp_2_coeff = comp_2_coeff/O2_coeff
    norm_CO2_coeff = CO2_coeff/O2_coeff
    norm_HF_coeff = HF_coeff/O2_coeff
    norm_H2O_coeff = H2O_coeff/O2_coeff
    norm_O2_coeff = O2_coeff/O2_coeff
    norm_N2_coeff = N2_coeff/O2_coeff

    stoic_eq = f'{comp_1_coeff}{formula_1} + {comp_2_coeff}{formula_2} + {O2_coeff} O2 + {N2_coeff} N2= {CO2_coeff} CO2 + {H2O_coeff} H2O + {HF_coeff} HF'

    norm_stoic_eq = f'{norm_comp_1_coeff}{formula_1} + {norm_comp_2_coeff}{formula_2} + {norm_O2_coeff} O2 + {norm_N2_coeff} N2= {norm_CO2_coeff} CO2 + {norm_H2O_coeff} H2O + {norm_HF_coeff} HF'


    LHS_C = comp_1_coeff*count_1.get('C', 0) + comp_2_coeff*count_2.get('C', 0) 
    LHS_H = comp_1_coeff*count_1.get('H', 0) + comp_2_coeff*count_2.get('H', 0) 
    LHS_F = comp_1_coeff*count_1.get('F', 0) + comp_2_coeff*count_2.get('F', 0) 
    LHS_O = O2_coeff*2

    
    RHS_C = CO2_coeff
    RHS_H = H2O_coeff*2 + HF_coeff
    RHS_F = HF_coeff
    RHS_O = CO2_coeff*2 + H2O_coeff

    assert LHS_C==RHS_C
    assert LHS_H==RHS_H
    assert LHS_F==RHS_F
    assert LHS_O==RHS_O

    print(stoic_eq)
    return norm_comp_1_coeff, norm_comp_2_coeff, stoic_eq, norm_stoic_eq


# Make binary blend input files and the run.sh 

In [44]:
#these are the pure components that my binary blends will feature
HCs = ["R50", "R290", "R600a"]
HFCs = ["R32", "R134a", "R152a"]


binary_blends = itertools.product(HCs,HFCs) 

for (comp1, comp2) in binary_blends:
    norm_comp_1_coeff, norm_comp_2_coeff, stoic_eq, norm_stoic_eq = find_stoichiometric_equation(comp1, comp2, (.5,.5))

    full_string = input_file_part_1

    species_block = f'''
species(
    label = '{comp1}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp1]['smiles']}')
)


species(
    label = '{comp2}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp2]['smiles']}')
)


species(
    label = 'O2',
    reactive = True,
    structure = SMILES('[O][O]')
)


species(
    label = 'N2',
    reactive = False,
    structure = SMILES('N#N')
)

'''
    full_string += species_block
    full_string += input_file_part_3
    
    simple_reactor_block = f"""
simpleReactor(
        temperature=[(800,'K'),(2000,'K')],
        pressure= [(1.0,'bar'),(10.0,'bar')],
        nSims=10,
        initialMoleFractions={{

        ##### stoichiometric combustion
        #normalized to oxygen: {norm_stoic_eq}
        #not normalized: {stoic_eq}



        "{comp1}": [{norm_comp_1_coeff-0.2*norm_comp_1_coeff}, {norm_comp_1_coeff+0.2*norm_comp_1_coeff}], #a little below and a little above stoich for a 50/50 binary blend
        "{comp2}": [{norm_comp_2_coeff-0.2*norm_comp_2_coeff}, {norm_comp_2_coeff+0.2*norm_comp_2_coeff}],
        "O2": 1,
        "N2": 3.76,
        

        }},
        #terminationRateRatio=1e-4,
        #terminationTime=(10,'s'),
        terminationTime=(2,'s'),
        #sensitivity=['halogen','OH'],
        #sensitivityThreshold=0.001,
        )

"""
    full_string += simple_reactor_block
    full_string += input_file_part_5

    #first, make the correct folder
    os.makedirs(f'./blends/{comp1}_{comp2}', exist_ok=True)
    
    with open(f'./blends/{comp1}_{comp2}/input.py', 'w') as f: 
        f.write(full_string)
    




0.5CH4 + 0.5CH2F2 + 1.5 O2 + 5.64 N2= 1.0 CO2 + 1.0 H2O + 1.0 HF
0.5CH4 + 0.5CH2F-CF3 + 1.75 O2 + 6.58 N2= 1.5 CO2 + 0.5 H2O + 2.0 HF
0.5CH4 + 0.5CH3-CHF2 + 2.25 O2 + 8.459999999999999 N2= 1.5 CO2 + 1.5 H2O + 1.0 HF
0.5C3H8 + 0.5CH2F2 + 3.0 O2 + 11.28 N2= 2.0 CO2 + 2.0 H2O + 1.0 HF
0.5C3H8 + 0.5CH2F-CF3 + 3.25 O2 + 12.219999999999999 N2= 2.5 CO2 + 1.5 H2O + 2.0 HF
0.5C3H8 + 0.5CH3-CHF2 + 3.75 O2 + 14.1 N2= 2.5 CO2 + 2.5 H2O + 1.0 HF
0.5C4H10 + 0.5CH2F2 + 3.75 O2 + 14.1 N2= 2.5 CO2 + 2.5 H2O + 1.0 HF
0.5C4H10 + 0.5CH2F-CF3 + 4.0 O2 + 15.04 N2= 3.0 CO2 + 2.0 H2O + 2.0 HF
0.5C4H10 + 0.5CH3-CHF2 + 4.5 O2 + 16.919999999999998 N2= 3.0 CO2 + 3.0 H2O + 1.0 HF


In [8]:
#if you have to re-iterate over this list, rerun this cell so the binary_blends variable is reset.

HCs = ["R50", "R290", "R600a"]
HFCs = ["R32", "R134a", "R152a"]


binary_blends = itertools.product(HCs,HFCs)

In [10]:
#write the SBATCH scripts required to run the RMG files

rmg_path = 'place the full path to your RMG install here, i.e. mine was /home/khalil.nor/Code/RMG-Py/rmg.py'


#feel free to change the SBATCH specifications as you desire
for (comp1, comp2) in binary_blends:
    print(comp1)

    run_string = f"""#!/bin/sh
    
#SBATCH --time=20-00:00:00
#SBATCH --job-name={comp1}_{comp2}_sic
#SBATCH --error=rmg.slurm.log
#SBATCH --output=output.slurm.log
#SBATCH --cpus-per-task=4
#SBATCH --mem-per-cpu=8Gb
#SBATCH --ntasks=1 
#SBATCH --partition=west


python {rmg_path} input.py

"""    
    
    
    with open(f'./single_component/species_in_core_blends/{comp1}_{comp2}/run.sh', 'w') as f: 
        f.write(run_string)

R50
R50
R290
R290
R290
R600a
R600a
R600a


# Single Components

Now we will generate the files for running RMG jobs of each single component. 

In [4]:

def find_stoichiometric_equation_single_comp(component_1): 

    
    dict_1 = atom_dictionaries[component_1]
    formula_1 = dict_1['formula']
    count_1 = dict_1['count']



    comp_1_coeff = 1

    
    CO2_coeff = comp_1_coeff*count_1.get('C', 0) 
    HF_coeff = comp_1_coeff*count_1.get('F', 0) 
    H2O_coeff = (comp_1_coeff*count_1.get('H', 0) - HF_coeff)/2
    O2_coeff = (2*CO2_coeff + H2O_coeff)/2
    N2_coeff = O2_coeff*3.76


    norm_comp_1_coeff = comp_1_coeff/O2_coeff 
    norm_CO2_coeff = CO2_coeff/O2_coeff
    norm_HF_coeff = HF_coeff/O2_coeff
    norm_H2O_coeff = H2O_coeff/O2_coeff
    norm_O2_coeff = O2_coeff/O2_coeff
    norm_N2_coeff = N2_coeff/O2_coeff

    stoic_eq = f'{comp_1_coeff}{formula_1} + {O2_coeff} O2 + {N2_coeff} N2= {CO2_coeff} CO2 + {H2O_coeff} H2O + {HF_coeff} HF'

    norm_stoic_eq = f'{norm_comp_1_coeff}{formula_1} + {norm_O2_coeff} O2 + {norm_N2_coeff} N2= {norm_CO2_coeff} CO2 + {norm_H2O_coeff} H2O + {norm_HF_coeff} HF'


    LHS_C = comp_1_coeff*count_1.get('C', 0)
    LHS_H = comp_1_coeff*count_1.get('H', 0) 
    LHS_F = comp_1_coeff*count_1.get('F', 0) 
    LHS_O = O2_coeff*2

    
    RHS_C = CO2_coeff
    RHS_H = H2O_coeff*2 + HF_coeff
    RHS_F = HF_coeff
    RHS_O = CO2_coeff*2 + H2O_coeff

    assert LHS_C==RHS_C
    assert LHS_H==RHS_H
    assert LHS_F==RHS_F
    assert LHS_O==RHS_O

    print(norm_stoic_eq)
    return norm_comp_1_coeff, stoic_eq, norm_stoic_eq


In [55]:
for comp1 in HCs+HFCs:
    norm_comp_1_coeff, stoic_eq, norm_stoic_eq = find_stoichiometric_equation_single_comp(comp1)
    
    if '+ -' in norm_stoic_eq: # there is a negative coefficient in the equation
        print(f""" Skipping {comp} because it needs COF2 to be stoichiometric, not yet implemented.""") # I was in a rush so I didn't automate this. 
        continue
    else: 


        full_string = input_file_part_1

        species_block = f'''
species(
    label = '{comp1}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp1]['smiles']}')
)


species(
    label = 'O2',
    reactive = True,
    structure = SMILES('[O][O]')
)


species(
    label = 'N2',
    reactive = False,
    structure = SMILES('N#N')
)
'''
    full_string += species_block
    full_string += input_file_part_3
    
    simple_reactor_block = f"""
simpleReactor(
        temperature=[(800,'K'),(2000,'K')],
        pressure= [(1.0,'bar'),(10.0,'bar')],
        nSims=10,
        initialMoleFractions={{

        ##### stoichiometric combustion
        #normalized to oxygen: {norm_stoic_eq}
        #not normalized: {stoic_eq}



        "{comp1}": [{norm_comp_1_coeff-0.2*norm_comp_1_coeff}, {norm_comp_1_coeff+0.2*norm_comp_1_coeff}], #a little below and a little above stoich for a 50/50 binary blend
        "O2": 1,
        "N2": 3.76,
        

        }},
        #terminationRateRatio=1e-4,
        #terminationTime=(10,'s'),
        terminationTime=(2,'s'),
        #sensitivity=['halogen','OH'],
        #sensitivityThreshold=0.001,
        )

"""
    full_string += simple_reactor_block
    full_string += input_file_part_5

    #first, make the correct folder
    os.makedirs(f'./single_component/{comp1}', exist_ok=True)
    
    with open(f'./single_component/{comp1}/input.py', 'w') as f: 
        f.write(full_string)
    


        

0.5CH4 + 1.0 O2 + 3.76 N2= 0.5 CO2 + 1.0 H2O + 0.0 HF
0.2C3H8 + 1.0 O2 + 3.7599999999999993 N2= 0.6 CO2 + 0.8 H2O + 0.0 HF
0.15384615384615385C4H10 + 1.0 O2 + 3.76 N2= 0.6153846153846154 CO2 + 0.7692307692307693 H2O + 0.0 HF
1.0CH2F2 + 1.0 O2 + 3.76 N2= 1.0 CO2 + 0.0 H2O + 2.0 HF
0.6666666666666666CH2F-CF3 + 1.0 O2 + 3.76 N2= 1.3333333333333333 CO2 + -0.6666666666666666 H2O + 2.6666666666666665 HF
 Skipping R152a because it needs COF2 to be stoichiometric
0.4CH3-CHF2 + 1.0 O2 + 3.7599999999999993 N2= 0.8 CO2 + 0.4 H2O + 0.8 HF


In [58]:
# make the sbatch files for the single component runs.
for comp1 in HCs+HFCs:
    print(comp1)

    run_string = f"""#!/bin/sh
    
#SBATCH --time=20-00:00:00
#SBATCH --job-name={comp1}
#SBATCH --error=rmg.slurm.log
#SBATCH --output=output.slurm.log
#SBATCH --cpus-per-task=4
#SBATCH --mem-per-cpu=8Gb
#SBATCH --ntasks=1 
#SBATCH --partition=west


python /home/khalil.nor/Code/RMG-Py/rmg.py input.py

"""    
    
    
    with open(f'./single_component/{comp1}/run.sh', 'w') as f: 
        f.write(run_string)

R50
R290
R600a
R32
R134a
R152a


# Make experimental cases and their run files 

In [10]:
#### R1234yf #################

atom_dictionaries = {'R1234yf': {'formula': 'CF3-CF=CH2', 
                                             'count': {'C': 3, 'F': 4, 'H': 2},
                                             'smiles': 'C=C(C(F)(F)F)F'},
                     "R32": {'formula': 'CH2F2', 'count': {'C': 1, 'H': 2, 'F': 2}, 'smiles': 'C(F)F'},
}

In [ ]:
comp1 = 'R1234yf'
                                            
# C3​H2​F4​+2.5O2​+9.4N2​→2CO2​+COF2​+2HF+9.4N2​
stoic_eq = 'C3​H2​F4​ + 2.5 O2 ​+ 9.4 N2 ​= 2 CO2 ​+ COF2​ + 2 HF + 9.4 N2​'
norm_stoic_eq = f'{1/2.5} C3​H2​F4​ + {2.5/2.5} O2 ​+ {9.4/2.5} N2 ​= {2/2.5} CO2 ​+ {1/2.5} COF2​ + {2/2.5} HF + {9.4/2.5} N2​'
norm_comp_1_coeff = 1/2.5

print(norm_stoic_eq)


full_string = input_file_part_1

species_block = f'''
species(
    label = '{comp1}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp1]['smiles']}')
)


species(
    label = 'O2',
    reactive = True,
    structure = SMILES('[O][O]')
)


species(
    label = 'N2',
    reactive = False,
    structure = SMILES('N#N')
)
'''
full_string += species_block
full_string += input_file_part_3

simple_reactor_block = f"""
simpleReactor(
        temperature=[(800,'K'),(2000,'K')],
        pressure= [(1.0,'bar'),(10.0,'bar')],
        nSims=10,
        initialMoleFractions={{

        ##### stoichiometric combustion
        #normalized to oxygen: {norm_stoic_eq}
        #not normalized: {stoic_eq}



        "{comp1}": [{norm_comp_1_coeff-0.2*norm_comp_1_coeff}, {norm_comp_1_coeff+0.2*norm_comp_1_coeff}], #a little below and a little above stoich for a 50/50 binary blend
        "O2": 1,
        "N2": 3.76,
        

        }},
        #terminationRateRatio=1e-4,
        #terminationTime=(10,'s'),
        terminationTime=(2,'s'),
        #sensitivity=['halogen','OH'],
        #sensitivityThreshold=0.001,
        )

"""
full_string += simple_reactor_block
full_string += input_file_part_5

#first, make the correct folder
os.makedirs(f'./experiment_cases/{comp1}', exist_ok=True)

with open(f'./experiment_cases/{comp1}/input.py', 'w') as f: 
    f.write(full_string)
    


        

In [15]:
######## R 32 / R 1234yf mixture ################

# 0.5 CH2​F2​ + 0.5 C3​H2​F4 ​+ 1.75 O2 ​+ 6.58 N2 ​→ 1.5 CO2 ​+ 0.5 COF2 ​+ 2.0 HF + 6.58 N2​

comp1 = "R32"
comp2 = 'R1234yf'


norm_comp_1_coeff = 0.5/1.75
norm_comp_2_coeff = 0.5/1.75 

stoic_eq = '0.5 CH2​F2​ + 0.5 C3​H2​F4 ​+ 1.75 O2 ​+ 6.58 N2 ​→ 1.5 CO2 ​+ 0.5 COF2 ​+ 2.0 HF + 6.58 N2​'
norm_stoic_eq = f'{0.5/1.75} CH2​F2​ + {0.5/1.75} C3​H2​F4 ​+ {1.75/1.75} O2 ​+ {6.58/1.75} N2 ​→ {1.5/1.75} CO2 ​+ {0.5/1.75} COF2 ​+ {2.0/1.75} HF + {6.58/1.75} N2​'

print(norm_stoic_eq)

full_string = input_file_part_1

species_block = f'''
species(
    label = '{comp1}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp1]['smiles']}')
)


species(
    label = '{comp2}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp2]['smiles']}')
)


species(
    label = 'O2',
    reactive = True,
    structure = SMILES('[O][O]')
)


species(
    label = 'N2',
    reactive = False,
    structure = SMILES('N#N')
)
'''
full_string += species_block
full_string += input_file_part_3

simple_reactor_block = f"""
simpleReactor(
        temperature=[(800,'K'),(2000,'K')],
        pressure= [(1.0,'bar'),(10.0,'bar')],
        nSims=10,
        initialMoleFractions={{

        ##### stoichiometric combustion
        #normalized to oxygen: {norm_stoic_eq}
        #not normalized: {stoic_eq}



        "{comp1}": [{norm_comp_1_coeff-0.2*norm_comp_1_coeff}, {norm_comp_1_coeff+0.2*norm_comp_1_coeff}], #a little below and a little above stoich for a 50/50 binary blend
        "{comp2}": [{norm_comp_2_coeff-0.2*norm_comp_2_coeff}, {norm_comp_2_coeff+0.2*norm_comp_2_coeff}],
        "O2": 1,
        "N2": 3.76,
        

        }},
        #terminationRateRatio=1e-4,
        #terminationTime=(10,'s'),
        terminationTime=(2,'s'),
        #sensitivity=['halogen','OH'],
        #sensitivityThreshold=0.001,
        )

"""
full_string += simple_reactor_block
full_string += input_file_part_5

    #first, make the correct folder
    # os.makedirs(f'./blends/{comp1}_{comp2}', exist_ok=True)
    
with open(f'./experiment_cases/{comp1}_{comp2}/input.py', 'w') as f: 
    f.write(full_string)





0.2857142857142857 CH2​F2​ + 0.2857142857142857 C3​H2​F4 ​+ 1.0 O2 ​+ 3.7600000000000002 N2 ​→ 0.8571428571428571 CO2 ​+ 0.2857142857142857 COF2 ​+ 1.1428571428571428 HF + 3.7600000000000002 N2​


In [21]:
atom_dictionaries = {"R50": {'formula': 'CH4', 'count': {'C': 1, 'H': 4}, 'smiles': 'C'}, 
                     "R290": {'formula': 'C3H8', 'count': {'C': 3, 'H': 8}, 'smiles': 'CCC'}, 
                     "R600a": {'formula': 'C4H10', 'count': {'C': 4, 'H': 10}, 'smiles': 'CC(C)C'}, 

                     
                     "R32": {'formula': 'CH2F2', 'count': {'C': 1, 'H': 2, 'F': 2}, 'smiles': 'C(F)F'},
                     "R134a": {'formula': 'CH2F-CF3', 'count': {'C': 2, 'H': 2, 'F': 4}, 'smiles': 'C(C(F)(F)F)F'},
                     "R152a": {'formula': 'CH3-CHF2', 'count': {'C': 2, 'H': 4, 'F': 2}, 'smiles': 'CC(F)F'},

                     "R1234ze(E)": {'formula': 'CHF=CH-CF3', 'count': {'C': 3, 'H': 2, 'F': 4}},
                     'R1234yf': {'formula': 'CF3-CF=CH2', 
                                             'count': {'C': 3, 'F': 4, 'H': 2},
                                             'smiles': 'C=C(C(F)(F)F)F'},
                     "HFO-1123": {'formula': 'CF2=CHF', 'count': {'C': 2, 'H': 1, 'F': 3}},

                     "CO2": {'formula': 'CO2', 'count': {'C': 1, 'O': 2}},
                     "HF": {'formula': 'HF', 'count': {'H': 1, 'F': 1}}, 
                     "H2O": {'formula': 'H2O', 'count': {'H': 2, 'O': 1}}, 
                    
                    }


binary_blends = [ #('R32', 'R50')
    ('R290', 'R32'),
    ('R290', 'R1234yf')
]

for (comp1, comp2) in binary_blends:
    norm_comp_1_coeff, norm_comp_2_coeff, stoic_eq, norm_stoic_eq = find_stoichiometric_equation(comp1, comp2, (.5,.5))

    full_string = input_file_part_1

    species_block = f'''
species(
    label = '{comp1}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp1]['smiles']}')
)


species(
    label = '{comp2}',
    reactive = True,
    structure = SMILES('{atom_dictionaries[comp2]['smiles']}')
)


species(
    label = 'O2',
    reactive = True,
    structure = SMILES('[O][O]')
)


species(
    label = 'N2',
    reactive = False,
    structure = SMILES('N#N')
)
'''
    full_string += species_block
    full_string += input_file_part_3
    
    simple_reactor_block = f"""
simpleReactor(
        temperature=[(800,'K'),(2000,'K')],
        pressure= [(1.0,'bar'),(10.0,'bar')],
        nSims=10,
        initialMoleFractions={{

        ##### stoichiometric combustion
        #normalized to oxygen: {norm_stoic_eq}
        #not normalized: {stoic_eq}



        "{comp1}": [{norm_comp_1_coeff-0.2*norm_comp_1_coeff}, {norm_comp_1_coeff+0.2*norm_comp_1_coeff}], #a little below and a little above stoich for a 50/50 binary blend
        "{comp2}": [{norm_comp_2_coeff-0.2*norm_comp_2_coeff}, {norm_comp_2_coeff+0.2*norm_comp_2_coeff}],
        "O2": 1,
        "N2": 3.76,
        

        }},
        #terminationRateRatio=1e-4,
        #terminationTime=(10,'s'),
        terminationTime=(2,'s'),
        #sensitivity=['halogen','OH'],
        #sensitivityThreshold=0.001,
        )

"""
    full_string += simple_reactor_block
    full_string += input_file_part_5

    #first, make the correct folder
    # os.makedirs(f'./blends/{comp1}_{comp2}', exist_ok=True)
    
    with open(f'./experiment_cases/{comp1}_{comp2}/input.py', 'w') as f: 
        f.write(full_string)
    




0.5C3H8 + 0.5CH2F2 + 3.0 O2 + 11.28 N2= 2.0 CO2 + 2.0 H2O + 1.0 HF
0.5C3H8 + 0.5CF3-CF=CH2 + 3.75 O2 + 14.1 N2= 3.0 CO2 + 1.5 H2O + 2.0 HF
